![Banner](../../../../Images/Banner/07_remote_sensing/09_03_banner_rs_rader.png)


# Sentinel-1 SAR Processing

The steps depend on the product (GRD or SLC) and what you want out of it (backscatter, coherence, deformation, polarimetry).


## Overview

The workflows below use ESA SNAP operator names since they are the most common reference, with Python equivalents where useful.


## Core concepts

### Learning objectives

1. Select scenes with one product type, orbit direction, relative orbit, mode, and polarisation, and a temporal baseline that fits the application.
2. Apply the GRD chain from precise orbits and noise removal through calibration, speckle filtering, terrain correction, and conversion to dB.
3. Coregister a GRD time series, apply a multitemporal speckle filter, and compute temporal features in linear units.
4. Prepare IW SLC data with TOPSAR-Split, an orbit file, back-geocoding, and enhanced spectral diversity.
5. Explain how DInSAR removes topographic phase, and how coherence, SBAS, and persistent scatterers use a stack differently.
6. Outline dual-pol decomposition, the EW sea-ice workflow, ship detection, and Level-2 ocean products.
7. Match a monitoring goal to a product and workflow, and name the GeoLattice functions for search, GRD preprocessing, RTC, and interferograms.

### Contents

1. Before processing: selecting scenes
2. GRD Backscatter Workflow (σ⁰ or γ⁰)
3. Multitemporal GRD time series workflow
4. SLC preprocessing (common steps for all phase-based work)
5. Differential InSAR (DInSAR) Workflow:  DInSAR vs. InSAR
6. Interferometric Coherence Workflow
7. InSAR Time Series (SBAS and PS-InSAR)
8. SBAS (Small Baseline Subset) Method
9. Persistent Scatterer (PS) InSAR
10. Dual-Pol Polarimetric Decomposition (SLC)
11. EW mode sea ice workflow
12. Ship detection workflow (GRD)
13. Level-2 OCN (Ocean) Products
14. Choosing the right workflow
15. Sentinel-1 Processing with GeoLattice
16. Summary and Conclusion
17. Additional Resources


## Before processing: selecting scenes

1. **Pick the product type**  
   - **GRD (Ground Range Detected):** Provides backscatter intensity data, useful for general land cover classification, crop mapping, flood detection, and ship/oil spill monitoring. It contains magnitude information only, not phase.
   - **SLC (Single Look Complex):** Contains both amplitude and phase information, required for applications like interferometry (InSAR for surface deformation), coherence analysis (change or stability detection), and advanced polarimetric decompositions. SLCs have finer spatial detail but require more complex processing, larger storage, and precise co-registration.

2. **Keep geometry consistent**  
   - **Orbit direction:** Use only ascending or only descending orbit scenes for a time series to ensure consistent viewing geometry. Mixing can cause undesirable geometry-related differences in backscatter.
   - **Relative orbit (track):** Use the same relative orbit number (track) when building multi-date stacks or time series, as this guarantees a nearly identical ground swath.
   - **Acquisition mode:** Use the same imaging mode. Interferometric Wide (IW) mode is standard for land; Extra Wide (EW) for sea ice; Stripmap (SM) for some specific sites.
   - **Frame/Burst alignment (for SLC):** For InSAR processing, ensure the scenes have the same burst/frame coverage and length to avoid edge artifacts or missing areas.

3. **Check polarization:**  
   - **VV+VH (vertical transmit/vertical receive + vertical transmit/horizontal receive):** Most common dual-pol for land surfaces, offering more information for crop types and land cover.
   - **HH+HV (horizontal transmit/horizontal receive + horizontal transmit/vertical receive):** Used mainly in polar or maritime regions, sensitive to sea ice and ocean phenomena.
   - Choose scenes with the same polarization(s) to facilitate comparison and stacking.

4. **Check temporal baseline:**  
   - **Coherence and InSAR applications:** Select pairs with the shortest possible time interval (6 or 12 days for Sentinel-1, depending on location and date). Shorter baselines minimize surface changes that corrupt interferometric phase and maximize the likelihood of high coherence.
   - **Time series analysis:** Consistent and regular revisit intervals improve quality and robustness of temporal statistics/results.

5. **Download precise orbit files (POEORB):**  
   - These files contain high-precision satellite position data, essential for accurate geocoding and InSAR. They are made available about 20 days after data acquisition.
   - For near-real-time applications, use restituted orbits (RESORB), which are less accurate but available within hours. For any scientific or mapping application, always rerun processing with POEORB as soon as they are published.

6. **Get a DEM (Digital Elevation Model):**  
   - A DEM is required for terrain correction and interferometric processing. The **Copernicus DEM (GLO-30, 30m resolution)** is currently the global standard, offering high accuracy and seamless coverage. It is freely available and recommended.
   - **SRTM 1 arc-second (30m):** Also works well for many areas, especially between latitudes 60°N and 56°S. For the best results in mountainous or non-SRTM regions, use GLO-30 or, if available, local high-resolution DEMs.


## GRD Backscatter Workflow (σ⁰ or γ⁰)

This workflow outlines the main steps needed to process Sentinel-1 GRD (Ground Range Detected) data in order to generate backscatter images. Backscatter (σ⁰ or γ⁰) gives information about how much of the radar signal is reflected from the ground back to the satellite, which is crucial for many land applications.

The process begins by reading the raw radar data and applying precise satellite position corrections. Next, it removes different types of noise, including instrument (thermal) noise and unwanted border artifacts. Afterward, the data is calibrated to turn raw digital numbers into physically meaningful backscatter values (σ⁰ or β⁰).

In areas with hills or mountains, terrain-flattening optionally adjusts for the effects of slope on the radar signal, producing γ⁰ backscatter that is normalized to local ground surface. To improve image quality, a speckle filter is applied to reduce radar "graininess" without losing important detail. 

Terrain correction then geocodes the image, aligning it with a real-world map projection and using a digital elevation model (DEM) to account for topography. Finally, the backscatter data is converted into decibel (dB) scale for easier interpretation and analysis, and subsets can be made for the area of interest.

This workflow ensures that the resulting images accurately reflect surface properties, are spatially consistent, and are ready for further analysis in GIS or remote sensing studies.

Used for: land cover, crop mapping, soil moisture, forest monitoring, flood mapping.

| Step | SNAP operator | Purpose and Mathematical Details |
|---|---|---|
| 1 | Read | Load the GRD `.SAFE` or `.zip` file, this contains the observed radar intensity data in Digital Numbers (DN) for each pixel. |
| 2 | Apply-Orbit-File | Refine the satellite’s known position by applying precise orbit corrections, to improve geolocation (coordinate mapping of radar returns). |
| 3 | ThermalNoiseRemoval | Remove additive instrument noise from the image, especially affecting cross-polarization (VH), typically by subtracting a noise floor estimated from calibration. If $I_{\text{raw}}$ is the raw intensity and $N$ is the thermal noise, then $I_{\text{clean}} = I_{\text{raw}} - N$. |
| 4 | Remove-GRD-Border-Noise | Mask out low-intensity scenes and artifacts typically present at the scene edges after focusing, often by thresholding the pixel intensity. |
| 5 | Calibration | Convert DN to backscatter coefficient ($\sigma^0$ or $\beta^0$). If $DN$ is the digital number and $CF$ is the calibration factor, $\beta^0 = DN^2 \times CF$. $\sigma^0$ is normalized radar cross section per unit ground area. Calibration ensures results can be physically interpreted and compared across scenes. |
| 6 | Terrain-Flattening (optional) | In hilly terrain, radar backscatter is affected by local slope. Convert $\beta^0$ to $\gamma^0$ to correct for this: $\gamma^0 = \frac{\beta^0}{\cos \theta_{local}}$, where $\theta_{local}$ is the local incidence angle (angle between radar and terrain normal). This produces estimates of radar backscatter as if the surface were flat, reducing topographic effects. |
| 7 | Speckle-Filter | Apply spatial filters (e.g., Refined Lee, Lee Sigma, Gamma-MAP) to reduce speckle noise, which is multiplicative. For example, the Lee filter's output at a pixel is: $I_{\text{filt}} = W \cdot I_{\text{mean}} + (1-W) \cdot I_{\text{orig}}$, where $W$ is a weighting function based on local mean and variance, and $I_{\text{orig}}$ is the original intensity. |
| 8 | Terrain-Correction | Geocode the raster, mapping radar image coordinates to map grid coordinates (e.g. UTM/WGS84) using a Digital Elevation Model (DEM) and Range-Doppler method, with pixel values resampled to a consistent ground resolution (typically 10 m). This ensures each pixel covers an equal-sized area on the ground. |
| 9 | LinearToFromdB | Convert the calibrated linear backscatter to decibel (dB) scale for analysis by $[\text{dB}] = 10 \times \log_{10}(\sigma^0)$, where $\sigma^0$ is the backscatter coefficient. The dB scale emphasizes relative differences and compresses dynamic range. |
| 10 | Subset / Write | Clip the image (spatial subset) to the Area of Interest (AOI) and export data layers as GeoTIFF for mapping and GIS analysis. |

**Notes**
- For terrain flattening, always calibrate to **$\beta^0$** (not $\sigma^0$), since $\beta^0$ is the radar reflectivity per slant range pixel, and only then apply the terrain-flattening correction to obtain **$\gamma^0$** for slope normalization.
- If multitemporal speckle filtering will be done later, skip single-date speckle filters here, multitemporal approaches reduce noise while preserving temporal signal.
- For all mathematical operations (averaging, composing, ratios, e.g. $r = \frac{VH}{VV}$), use the linear (not dB) backscatter values throughout the workflow. Only convert to dB after all calculations to avoid bias, since $10\log_{10}(\text{mean}(x)) \neq \text{mean}(10\log_{10}(x))$.
- Statistical and band derivations after terrain correction (e.g., band ratio $VH/VV$ or Radar Vegetation Index $RVI = \frac{4 \cdot VH}{VV + VH}$) should be computed using the linear backscatter values (not dB).


## Multitemporal GRD time series workflow

Used for: crop phenology, rice mapping, change detection, and deforestation monitoring using multi-temporal Sentinel-1 GRD data.

Multitemporal GRD analysis leverages the temporal dimension to investigate surface change and seasonal dynamics. Each step is critical to ensuring spatial and radiometric consistency throughout the time series, enabling reliable extraction of temporal features per pixel.

1. **Run steps 1 to 5 of the GRD workflow on each scene:**  
   - Steps include: reading data, applying precise orbit files, removing thermal noise, masking border noise, and calibrating to backscatter coefficients (β⁰ or σ⁰) for each image.  
   - All images should be from the same acquisition geometry (same Sentinel-1 satellite, track, frame, and orbit direction) to ensure spatial alignment and comparable incidence angles.

2. **Coregister the stack:**  
   - Temporal co-registration ensures all images align perfectly at the pixel level over the time series.  
   - Use SNAP `CreateStack` operator for precise co-registration of all scenes (can also terrain-correct each image to the same projected grid, which is often sufficient for land cover analysis).

3. **Apply multitemporal speckle filtering:**  
   - Use SNAP's `Multi-Temporal-Speckle-Filter` (e.g., Quegan filter) to reduce speckle while preserving both spatial structures and the temporal signature in each pixel.  
   - Multitemporal filters outperform single-image filters by averaging over time, assuming temporal consistency of underlying signals.

4. **Terrain correction on common grid:**  
   - Apply terrain correction to all images so they are geocoded to a consistent map grid (such as UTM), using a DEM (e.g., Copernicus GLO-30).  
   - This step compensates for geometric distortions due to topography and ensures that multi-date stacking and later spatial analysis are reliable.

5. **Convert to dB:**  
   - Transform the calibrated, terrain-corrected backscatter from linear power ($\sigma^0$ or $\gamma^0$) to decibel scale: [dB] $= 10 \log_{10}(\sigma^0)$ per pixel.  
   - dB values are easier to interpret for change detection, providing an intuitive measure of relative differences.

6. **Build a data cube:**  
   - Stack all the processed images into a multi-temporal data cube, such as an `xarray.DataArray`, where the time dimension indexes dates of acquisition and spatial dimensions cover the Area of Interest.  
   - This enables pixelwise analysis across time and facilitates downstream feature extraction and modeling.

7. **Extract temporal features per pixel:**  
   - Calculate statistics such as mean, median, standard deviation, minimum, maximum, and percentiles for each pixel across the time axis.  
   - Application-specific features include, for example, the date of minimum VH (useful for rice transplanting/flood detection), amplitude of seasonal change, and the speed or slope of temporal backscatter increase.

8. **Optional smoothing:**  
   - To further reduce residual noise in the temporal signal, apply temporal smoothing filters (e.g., Savitzky-Golay filter or a running mean) to each pixel’s time series.  
   - This is especially useful before extracting features sensitive to noise, such as change point detection.

9. **Classification or modeling:**  
   - Use the extracted temporal features as input variables for land cover classification or change detection models.  
   - Common algorithms include Random Forest, XGBoost, or, for time series sequence modeling, deep learning approaches like LSTM or Transformer architectures.

10. **Validation:**  
    - Assess the accuracy of classification or change maps using ground truth from field surveys, agricultural records, or high-resolution reference imagery.

**Rice mapping tip:**  
- Rice transplanting causes a sudden decline in VH backscatter (flooded, bare soil; often below −20 dB), followed by a progressive increase as plants grow and scatter more radar energy. Detecting this sharp VH minimum and subsequent rise is a robust temporal indicator for paddy rice phenology.


## SLC preprocessing (common steps for all phase-based work)

Sentinel-1 IW SLC (Single Look Complex) data are divided into 3 subswaths (IW1, IW2, IW3), and each subswath contains about 9 bursts. Each burst is an individual segment of data acquired in time and must be handled separately to avoid introducing artifacts, especially in phase analysis (e.g., for InSAR).

### SLC Preprocessing Steps

| Step | SNAP operator | Purpose | Mathematical/Technical Explanation |
|---|---|---|---|
| 1 | TOPSAR-Split | Select subswath(s), polarization(s), and only the bursts covering your AOI (area of interest) | Isolates subsets of data: If $S(t, s)$ is the signal as a function of time $t$ and subswath $s$, this step selects $S(t, s_{\mathrm{desired}})$ and spatially subsets by burst to your AOI. |
| 2 | Apply-Orbit-File | Applies precise orbital information | The satellite’s actual position and velocity can be denoted by $\vec{r}(t), \vec{v}(t)$. This step refines the navigation metadata so pixel geolocation produces correct phase and amplitude geometry, reducing errors $\Delta \vec{r}$ introduced by the predicted orbit. Accurate orbits are crucial because any orbital error $\delta \vec{r}$ leads to phase errors $\Delta \phi \approx -\frac{4\pi}{\lambda} (\delta \vec{r} \cdot \hat{l})$ where $\lambda$ is radar wavelength and $\hat{l}$ is the look vector. |
| 3 | Back-Geocoding | Coregisters secondary (slave) image to reference (master) using precise orbits and a DEM (Digital Elevation Model) | Each SLC pixel in the reference image has coordinates $(x^{(ref)}_i, y^{(ref)}_i)$; the secondary needs to be resampled so corresponding pixels align spatially. Formally, find a mapping $T$ such that $(x^{(slave)}_j, y^{(slave)}_j) \to (x^{(ref)}_i, y^{(ref)}_i)$. Back-geocoding thus matches each pixel in the slave to the reference, correcting for geometry and topography to minimize spatial misalignment, which is critical for consistent phase comparison. |
| 4 | Enhanced-Spectral-Diversity (ESD) | Fine azimuth (along-track) coregistration; essential for TOPS to avoid phase jumps at burst boundaries | TOPS mode SLCs are acquired by rotating the beam during acquisition, which can create phase discontinuities at burst edges if bursts are not precisely aligned. Spectral diversity uses the overlap regions between bursts to estimate and correct small coregistration errors. The phase correction, $\phi_{\text{corr}}$, is estimated using the difference between overlapping burst spectra, minimizing phase discontinuities by adjusting the azimuth alignment, i.e., $\phi_{\text{ESD}} = \arg \left ( \sum_{k} S_1(k) S_2^*(k) \right )$, where $S_1(k)$ and $S_2(k)$ are the Fourier components in the overlap and $^*$ denotes complex conjugation. |

After these 4 preprocessing steps (TOPSAR-Split, Apply-Orbit-File, Back-Geocoding, and ESD), the coregistered SLCs are ready for advanced SAR processing, such as interferometry (measuring displacement/deformation from phase $\phi$), coherence (computing temporal or spatial correlation), or polarimetry (analyzing scattering mechanisms by different polarizations).


## Differential InSAR (DInSAR) Workflow:  DInSAR vs. InSAR

**What is DInSAR and how does it differ from conventional InSAR?**

- **InSAR (Interferometric Synthetic Aperture Radar):** Uses the phase difference $\Delta\phi$ between two Synthetic Aperture Radar (SAR) images acquired from slightly different viewpoints (typically a short time apart, or in a “single-pass” configuration) to map topography. This “topographic phase” encodes the surface elevation.
- **DInSAR (Differential InSAR):** Measures *surface deformation over time* by comparing two SAR images acquired at different times (from slightly different orbital positions). DInSAR removes the constant topographic phase (using a DEM or a third SAR acquisition, see below), so the remaining phase reflects only surface movement (deformation), atmospheric delays, and noise.

**Key difference:**  
- *InSAR measures topography; DInSAR measures temporal ground motion by removing topographic contribution.*

**Mathematical details**  

The observed interferometric phase at a pixel can be modeled as:

$\displaystyle \phi_{\text{total}} = \phi_{\text{flat}} + \phi_{\text{topo}} + \phi_{\text{disp}} + \phi_{\text{atm}} + \phi_{\text{noise}}$

where:
- $\phi_{\text{flat}}$: flat-earth phase (due to imaging geometry)
- $\phi_{\text{topo}}$: phase due to surface elevation (topography)
- $\phi_{\text{disp}}$: phase due to displacement (the signal DInSAR seeks)
- $\phi_{\text{atm}}$: phase delays from atmosphere
- $\phi_{\text{noise}}$: decorrelation and noise

In *conventional InSAR*, the primary outcome is $\phi_{\text{topo}}$ (topography).  
In *DInSAR*, we wish to compute $\phi_{\text{disp}}$ (displacement), which requires removing both $\phi_{\text{flat}}$ and $\phi_{\text{topo}}$ (using accurate orbits and a DEM).

The line-of-sight (LOS) displacement $d_{\text{LOS}}$ is related to unwrapped phase as:

$\displaystyle d_{\text{LOS}} = -\frac{\lambda}{4\pi} \, \phi_{\text{unwrapped}}$

where:
- $\lambda$ is the radar wavelength (here, $\approx 5.55 \,\text{cm}$ for Sentinel-1),
- $\phi_{\text{unwrapped}}$ is the phase after flattening, topography removal, and phase unwrapping.

*Example*: For Sentinel-1, one full fringe (phase cycle of $2\pi$) corresponds to $\frac{\lambda}{2}$, so $\approx 2.8$ cm of LOS motion.

**DInSAR Workflow Steps with Explanations**

| Step | SNAP operator | Mathematical/Technical Purpose |
|---|---|---|
| 1 to 4 | (SLC preprocessing) | See earlier: split SLC, orbit refinement, coregistration, spectral diversity to align images and ensure pixel correspondence. (Math: aligns $S_1(x, y)$ to $S_2(x, y)$ for pixelwise phase difference). |
| 5 | Interferogram | Compute complex interferogram: $I = S_{\mathrm{ref}} \cdot S^{*}_{\mathrm{sec}}$; phase $\phi = \arg(I)$. Remove $\phi_{\text{flat}}$ for proper phase referencing; compute coherence as $|\langle S_{\mathrm{ref}} S^{*}_{\mathrm{sec}}\rangle|/(\sigma_{\mathrm{ref}}\sigma_{\mathrm{sec}})$ to assess reliability. |
| 6 | TOPSAR-Deburst | Combine individual bursts into a continuous SLC grid (needed for Sentinel-1 IW mode, preserves phase consistency across bursts). |
| 7 | TOPSAR-Merge (if needed) | Mosaic subswaths if AOI spans multiple sub-images. |
| 8 | TopoPhaseRemoval | Subtract computed topographic phase $\phi_{\text{topo}}$ using a DEM and precise baseline; isolates $\phi_{\text{disp}}$. |
| 9 | Multilook | Spatial averaging to reduce speckle and yield near-square pixels (e.g., $4 \times 1$ looks combines pixels in range and azimuth). |
| 10 | Goldstein-Phase-Filtering | Apply adaptive filtering to smooth random phase noise while preserving deformation signals. |
| 11 | SNAPHU export → unwrapping → import | "Unwrap" phase so $\phi \in \mathbb{R}$, not just $[-\pi, \pi]$. (Mathematically, integrate phase jumps to reconstruct continuous phase field.) |
| 12 | Phase-To-Displacement | Apply $d_{\text{LOS}} = -(\lambda/4\pi)\phi_{\text{unwrapped}}$ to convert phase into meters of surface motion. |
| 13 | Terrain-Correction | Map output to geographic coordinates (orthorectification). |

**Quality checks and best practices**
- *Coherence masking*: Exclude pixels with low coherence (typically $<$0.3), as noisy or rapidly changing ground conditions cause decorrelation.
- *Baseline control*: Use pairs with low perpendicular baseline to reduce spatial decorrelation and minimize topographic errors.
- *Atmospheric effects*: Compare phase with topography and/or meteorological data to spot and correct for stratified atmospheric delays.
- *Multi-track processing*: To resolve 2D or 3D motion, combine DInSAR from ascending/descending passes; invert for up-/east-/north-components.

**Summary Table, InSAR vs. DInSAR:**  
|  | InSAR | DInSAR |
|--|-------|--------|
| Main Output | Surface elevation (DEM) | Surface deformation (displacement map) |
| Removes topography? | No | Yes, using DEM or 3-image stack |
| Key Math | $\phi \approx \phi_{\text{flat}} + \phi_{\text{topo}}$ | $\phi_{\text{disp}} = \phi - \phi_{\text{flat}} - \phi_{\text{topo}}$ |
| Typical application | DEM generation | Monitoring earthquakes, volcanoes, landslides, urban subsidence |

**References:**  
- [Massonnet & Feigl, "Radar interferometry and its application to changes in the Earth’s surface," Rev. Geophys., 1998.](https://doi.org/10.1029/97RG03139)  
- [ESA Sentinel-1 InSAR Guide.](https://sentinel.esa.int/web/sentinel/user-guides/sentinel-1-sar/interferometry)


## Interferometric Coherence Workflow

Interferometric coherence is a measure of how similar two radar images (SLCs, or Single Look Complex data) are when observed at two different times. It quantifies, on a pixel-by-pixel basis, the degree to which ground features remain unchanged (“correlated”) versus altered between the two acquisitions.

The coherence workflow typically involves preparing the SLC data (splitting bursts, aligning the images, correcting orbits), then computing the local similarity using a mathematical formula that compares the two radar signals over a small neighborhood or moving window. This results in a new image showing, for each pixel, how stable or reliable the radar signal is between the two dates, a value from 0 (completely different, very unstable or changed ground) to 1 (identical, extremely stable).

Coherence maps are crucial for applications such as change detection, damage assessment after disasters, monitoring subsidence, or even tracking agricultural activity, because areas that experience change (like plowed soil, flooded fields, or collapsed buildings) will show a marked drop in coherence, while unchanged ground remains highly coherent.

**Applications:** Crop harvesting and tillage detection, urban mapping, flood extent in built-up areas, earthquake damage mapping, and more.

The interferometric coherence $\gamma$ quantifies the similarity (correlation) between two complex SAR images (typically Single Look Complex, SLCs) acquired at different times. Coherence is computed as:

$$
\gamma = \frac{|\langle S_{\text{ref}} S^*_{\text{sec}} \rangle|}{\sqrt{ \langle |S_{\text{ref}}|^2 \rangle \langle |S_{\text{sec}}|^2 \rangle }}
$$

where:
- $S_{\text{ref}}$ is the reference complex SAR image,
- $S_{\text{sec}}$ is the secondary image,
- $S^*_{\text{sec}}$ is the complex conjugate of the secondary image,
- $\langle \cdot \rangle$ denotes spatial averaging over a moving window (e.g., 10 pixels in range × 3 pixels in azimuth).

The denominator normalizes the correlation (so that $0 \leq \gamma \leq 1$): 
- $\gamma=1$ indicates perfect correlation (no change, very stable scatterers).
- $\gamma=0$ indicates complete decorrelation (temporal, volume, or geometric change).

**Step-by-step workflow:**

1. **SLC preprocessing**: Split subswaths, apply precise orbits, perform back-geocoding, and (with Sentinel-1 IW data) Enhanced Spectral Diversity (ESD) to align the images and ensure pixel correspondence. Mathematically, this ensures that $S_{\text{ref}}(x, y)$ and $S_{\text{sec}}(x, y)$ represent the same ground location.

2. **Coherence operator:** Calculate $\gamma$ for each pixel using a moving window (typically $10 \times 3$ pixels) as above. Spatial averaging both improves the stability of $\gamma$ and mitigates speckle noise, but at a loss of spatial detail.

3. **TOPSAR-Deburst:** For Sentinel-1 IW, SLC data are collected in bursts; de-bursting stacks them together, ensuring continuity of phase and coherence across burst boundaries.

4. **TOPSAR-Merge (if applicable):** If the Area of Interest (AOI) spans multiple sub-swaths, these are mosaicked into one continuous image.

5. **Multilook (optional):** Further spatial averaging (“looks”) can be applied to reduce speckle and improve radiometric quality. Mathematically, multilooking performs local averaging of both amplitude and phase, which reduces the variance of the coherence estimate while degrading resolution.

6. **Terrain Correction:** The resulting coherence (and other SAR products) are geocoded via orthorectification, mapping image coordinates to map coordinates using a DEM and the satellite's geometry.

7. **Build coherence time series:** By pairing consecutive SLC scenes (e.g., 6 or 12 days apart for Sentinel-1), and repeating the above, a time series $\gamma(t)$ can be assembled for each pixel/patch. Time variations in $\gamma$ can then be interpreted for land cover change, agricultural practices, surface stability, or post-disaster damage.

**Physical interpretation:**
- $\gamma \approx 1$: Very stable scatterers (e.g., buildings, rocks), little or no phase change.
- $\gamma \ll 1$: Significant decorrelation due to rapid ground cover change (e.g., vegetation growth, plowing, flooding, snow, or urban destruction).

**Practical note:** A sharp drop in coherence over urban/built-up regions after an earthquake often indicates structural damage, as randomization of scatterers causes decorrelation.


## InSAR Time Series (SBAS and PS-InSAR)

**Purpose:** Detect and quantify long-term surface motions (e.g., subsidence from groundwater withdrawal, mining, or infrastructure shifts) with millimeter-per-year (mm/yr) accuracy.

**Data Requirement:** At least 20 Sentinel-1 SLC (Single Look Complex) scenes acquired from the same satellite track, spanning a year or longer. These images provide phase information essential for temporal analysis.


## SBAS (Small Baseline Subset) Method

The SBAS method is a popular InSAR (Interferometric SAR) time series technique designed to track slow, long-term ground movements, such as subsidence, uplift, or landslide, with high accuracy. It works by carefully selecting pairs of radar images that are close together in both acquisition time and satellite position (“small baselines”), which helps to minimize errors from atmospheric variations and differences in imaging geometry. By analyzing a large set of these tightly spaced interferograms over time, SBAS can create displacement time-series for every pixel, helping scientists monitor ground deformation at millimeter precision over years.

**1. Download SLC Data:** Obtain $N$ SLC images $\{S_1, S_2, \dots, S_N\}$ and the corresponding precise orbit ephemerides.

**2. Coregistration:** All SLCs are geometrically aligned (coregistered) to a common reference date (say, $S_{ref}$) so that pixels correspond spatially: $S_k(x, y)$ aligns to ground location $(x, y)$ across all $k$.

**3. Interferogram Network Construction:**  
Form a set of interferometric pairs $(i, j)$ such that both their temporal baseline $\Delta T_{ij}$ and perpendicular spatial baseline $\Delta B_\perp^{ij}$ are below chosen thresholds. The network is typically sparse to avoid large decorrelation.

**4. Interferogram Generation & Unwrapping:**  
For each selected $(i, j)$:
- Compute the wrapped phase difference:
  $$
  \phi_{ij}(\mathbf{r}) = \arg\left[ S_i(\mathbf{r}) S_j^*(\mathbf{r}) \right]
  $$
  where $*$ denotes complex conjugate.
- Apply filtering and unwrap phases in space to obtain $\tilde{\phi}_{ij}(\mathbf{r})$.

**5. Network Inversion for Displacement:**  
For a stack of $N$ images, each interferogram measures relative phase changes:
  $$
  \tilde{\phi}_{ij} (\mathbf{r}) = \phi_j(\mathbf{r}) - \phi_i(\mathbf{r}) + \delta_{ij}(\mathbf{r})
  $$
where $\delta_{ij}$ encompasses atmospheric, orbital, and noise contributions.
Solve the system of equations (one per interferogram) for the phase (displacement) at each acquisition, often via least-squares or singular value decomposition (SVD) inversion. The phase is related to displacement as:
  $$
  d_k(\mathbf{r}) = \frac{\lambda}{4\pi} \, \phi_k(\mathbf{r})
  $$
where $\lambda$ is the radar wavelength.

**6. Atmospheric Delay Correction:**  
Remove atmospheric artifacts by estimating $\delta_{ij}$ (with external models, e.g. GACOS or ERA5, or spatial filtering).

**7. DEM Error Correction:**  
The measured phase comprises a DEM error term; estimate a constant DEM offset for each pixel or scene and subtract it.

**8. Velocity and Displacement Time Series:**  
The per-date displacement estimates $d_k(\mathbf{r})$ yield a time series for every ground pixel; from the linear fit, estimate velocity:
  $$
  v(\mathbf{r}) = \frac{d_{N}(\mathbf{r}) - d_1(\mathbf{r})}{T_{N} - T_1}
  $$
where $T_k$ is the time of scene $k$.

**9. Reference Point Calibration:**  
Remove residual phase bias by "zeroing" the displacement to a non-moving reference pixel or calibrate against a GNSS station.


## Persistent Scatterer (PS) InSAR

PS-InSAR stands for "Persistent Scatterer Interferometric Synthetic Aperture Radar." 
It's an advanced technique that uses a stack of SAR images to precisely measure ground movement at individual, stable point scatterers (such as buildings, rocks, or other objects that remain consistent over time) with millimeter accuracy.

The method works by analyzing changes in the phase of reflected radar signals across many satellite passes, focusing only on those locations ("persistent scatterers") that behave very consistently over time. By looking at these stable reflection points, PS-InSAR can filter out much of the noise and atmospheric effects that limit conventional InSAR, allowing for long-term ground deformation time series, even in areas with lots of vegetation or urban development.

1. **Coregistration:** All Single Look Complex (SLC) images $S_k$ ($k=1\dots N$) are coregistered to a common reference scene $S_{ref}$, so that pixels correspond spatially across the stack: $S_k(x, y)$ refers to the same ground location $(x, y)$ in all scenes.

2. **Forming Interferograms with a Common Reference:** For each subsequent SLC image $S_k$ ($k \neq ref$), form the interferogram with respect to the reference:
   $$
   I_k(\mathbf{r}) = S_k(\mathbf{r}) S_{ref}^*(\mathbf{r})
   $$
   where $\mathbf{r}$ denotes pixel location, and $^*$ denotes the complex conjugate. This yields $N - 1$ interferograms, all referenced to the same epoch.

3. **Selection of Persistent Scatterers:** For each pixel, compute the amplitude dispersion index:
   $$
   D_{amp}(\mathbf{r}) = \frac{\sigma_A(\mathbf{r})}{\mu_A(\mathbf{r})}
   $$
   where $\sigma_A$ is the standard deviation and $\mu_A$ is the mean of the amplitude time series at $\mathbf{r}$. Pixels with $D_{amp}$ below a threshold (commonly $0.25$) are considered Persistent Scatterers (PS). Alternatively, phase stability or spectral tests can be used.

4. **Parameter Estimation for PS Points:** For each PS pixel, the unwrapped interferometric phase $\tilde{\phi}_k(\mathbf{r}_{PS})$ (relative to the reference) is modeled as:
   $$
   \tilde{\phi}_k(\mathbf{r}_{PS}) = \phi_{disp}(t_k) + \phi_{DEM} + \phi_{atm}(t_k) + \phi_{noise}(t_k)
   $$
   - $\phi_{disp}$: phase due to displacement (to be estimated, typically linear in time for velocity, plus possible nonlinear components).
   - $\phi_{DEM}$: phase due to DEM errors (constant per pixel).
   - $\phi_{atm}$: atmospheric phase (varies per image).
   - $\phi_{noise}$: residual noise.
   Parameter estimation is commonly done by time series analysis (e.g., Least Squares, SVD). The linear component yields mean velocity:
   $$
   v_{PS} = \frac{\lambda}{4\pi} \cdot \text{slope}\left[\tilde{\phi}_k\right]
   $$
   where $\lambda$ is radar wavelength.

5. **Result:** Output is a set of point-wise displacement time series $d_k(\mathbf{r}_{PS})$ and mean velocities $v_{PS}$ for each detected persistent scatterer.

**Tools:** ISCE2 `topsStack` + MintPy (SBAS for area-based, PS for point-based), SNAP → StaMPS (PS-InSAR), ASF HyP3 + MintPy (cloud), Dolphin (phase linking), NASA OPERA DISP-S1 (North America ready-made).


## Dual-Pol Polarimetric Decomposition (SLC)

Dual-pol polarimetric decomposition is a technique used to extract detailed information about the physical and structural properties of the Earth's surface from Synthetic Aperture Radar (SAR) data, specifically when only two polarization channels- usually VV (vertical transmit, vertical receive) and VH (vertical transmit, horizontal receive) -are available. 

In polarimetric SAR (PolSAR), each pixel of an image can contain complex information (amplitude and phase) about how radar signals interacted with the Earth's surface. Dual-pol decomposition focuses on mathematically analyzing the relationships between the two available polarimetric channels (VV and VH), allowing us to infer land cover characteristics, detect and classify vegetation, map wetlands, and even gauge crop or forest structure.

The process involves creating a $2\times2$ complex covariance matrix ($\mathbf{C}_2$) at each pixel by combining the VV and VH channels. This matrix encodes both the intensity and the correlation between the two polarizations. Polarimetric decomposition methods (such as H/α decomposition) then analyze the eigenvalues and eigenvectors of this matrix to derive physically-interpretable parameters: entropy ($H$), which measures randomness or heterogeneity of scattering; anisotropy ($A$), which contrasts the strengths of different scattering mechanisms; and the mean alpha angle ($\alpha$), which indicates the dominant type of scattering (surface, volume, or double-bounce) at that location.

By applying dual-pol decomposition to Sentinel-1 SLC data, we can generate thematic maps that highlight differences in land cover, identify flooded areas, monitor vegetation change, or characterize surface structure, all using only the information in the VV and VH channels. This enables many advanced remote sensing applications even when full quad-polarimetric data is not available.

Used for: land cover classification, vegetation structure, wetland mapping.

The following workflow enables extraction of physical and structural properties of the scene using only the two dual-polarization channels (VV and VH) available in Sentinel-1 SLC data:

1. **TOPSAR-Split**: Split both VV and VH polarization channels from the SLC product, so that $S_{VV}(x, y)$ and $S_{VH}(x, y)$ are available for all pixels $(x, y)$.

2. **Apply-Orbit-File**: Apply precise orbit information to correct satellite position and timing.

3. **Calibration with complex output**: Perform radiometric calibration to obtain calibrated backscatter values while preserving both amplitude and phase (complex values) of $S_{VV}$ and $S_{VH}$ for each pixel.

4. **TOPSAR-Deburst**: Merge burst-wise SLC slices into a single, seamless scene for further processing.

5. **Polarimetric-Matrices (C2 covariance matrix)**:
   For dual-pol (VV,VH), form the $2 \times 2$ complex covariance (C2) matrix for each pixel:
   $$
   \mathbf{C}_2 = 
   \begin{bmatrix}
     \langle S_{VV} S_{VV}^* \rangle & \langle S_{VV} S_{VH}^* \rangle \\
     \langle S_{VH} S_{VV}^* \rangle & \langle S_{VH} S_{VH}^* \rangle
   \end{bmatrix}
   $$
   where $\langle \cdot \rangle$ denotes local multilook or averaging.

6. **Multilook**: Spatial averaging (multilooking) improves radiometric quality and reduces speckle, so all covariance/clutter statistics are estimated over a $n \times m$ pixel window:
   $$
   \langle S_{ij} S_{kl}^* \rangle = \frac{1}{N} \sum_{n=1}^N S_{ij}^{(n)} \left(S_{kl}^{(n)}\right)^*
   $$

7. **Polarimetric-Speckle-Filter** (e.g. Refined Lee): Apply a polarimetric-adapted speckle filter to the C2 matrix to suppress noise while preserving polarimetric information.

8. **Polarimetric-Decomposition → H-Alpha dual-pol**: 
   Perform eigenvalue/eigenvector decomposition of the (filtered) C2 matrix for each pixel, yielding the polarimetric entropy $H$, anisotropy $A$, and alpha angle $\alpha$:
   - Let $\lambda_1, \lambda_2$ be the normalized eigenvalues of $C_2$ ($\lambda_1 + \lambda_2 = 1$), then:
     - **Entropy:**
       $$
       H = -\sum_{i=1}^2 \lambda_i \log_2 \lambda_i
       $$
     - **Anisotropy:**
       $$
       A = \frac{\lambda_1 - \lambda_2}{\lambda_1 + \lambda_2}
       $$
     - **Alpha angle:** Weighted mean of scattering angle, from the eigenvector orientations; in dual-pol:
       $$
       \alpha = \sum_{i=1}^2 \lambda_i \alpha_i
       $$
     where $\alpha_i$ is derived from the eigenvector corresponding to $\lambda_i$.
   These parameters help distinguish different scattering mechanisms (e.g. surface, volume, dihedral) via physical interpretation.

9. **Terrain-Correction**: Geocode the results into map coordinates accounting for local terrain/topography.

10. **Feature extraction and classification**: The derived features $H$ (entropy), $A$ (anisotropy), $\alpha$ (alpha angle) at each pixel are used as inputs to a classification algorithm (e.g., Random Forest, SVM, or unsupervised clustering) for land cover or structure identification.

> **Note:** Since Sentinel-1 only acquires VV and VH, the C2 (dual-pol) and its associated decompositions (H-Alpha dual-pol) are used, rather than full quad-pol decompositions (e.g., Freeman-Durden, Yamaguchi) which require HH and HV as well.


## EW mode sea ice workflow

1. **Read EW GRDM (Ground Range Detected Medium resolution) bands (HH, HV):**
   - Obtain two polarization channels: $S_{HH}$ and $S_{HV}$, where $S_{pq}$ denotes the complex SAR signal for transmit $p$ and receive $q$ polarization.

2. **Apply Orbit File:**
   - Refine the satellite's position and velocity by incorporating the precise orbit ephemeris. This improves geolocation accuracy of each pixel.

3. **Thermal Noise Removal:**
   - Remove additive noise from each pixel, especially important for the HV channel over sea ice. For each pixel:
     $$
     S_{corr} = S_{raw} - N_{thermal}
     $$
   - Where $S_{raw}$ is the observed signal, $N_{thermal}$ the estimated thermal noise, and $S_{corr}$ the corrected value.

4. **Radiometric Calibration to $\sigma^0$:**
   - Convert DN (digital number) to calibrated backscatter coefficient $\sigma^0$:
     $$
     \sigma^0 = \mathrm{CF} \cdot |S_{corr}|^2
     $$
   - Where $\mathrm{CF}$ is a calibration factor (depends on the product and sensor).

5. **Incidence Angle Correction:**
   - EW mode has a large incidence angle range ($\theta \approx 19^\circ$ to $47^\circ$). Correct $\sigma^0$ to a reference angle $\theta_{ref}$ (e.g., $35^\circ$):
     $$
     \sigma^0_{ref} = \sigma^0_{obs} + f(\theta_{ref}) - f(\theta_{obs})
     $$
   - Where $f(\theta)$ models the angular dependence, often empirically fit as $f(\theta) = a\,\theta + b\,\theta^2$.

6. **Speckle Filtering:**
   - Apply a spatial filter (e.g. Lee or Frost) over a moving window of $n \times m$ pixels to suppress speckle:
     $$
     \hat{\sigma}^0_{i,j} = \frac{1}{N} \sum_{k=1}^N \sigma^0_{i_k,j_k}
     $$
   - Where $\hat{\sigma}^0_{i,j}$ is the filtered backscatter at $(i,j)$.

7. **Ellipsoid/Terrain Correction (Geocoding):**
   - Map the image to a cartographic projection (e.g., polar stereographic projection, EPSG:3413), correcting for topography:
     $$
     (x,y)_{\text{map}} = f(\text{lat}(\text{range}, \text{az}), \text{lon}(\text{range}, \text{az}))
     $$

8. **Texture Feature Extraction (GLCM) and Band Ratios:**
   - Compute statistical texture features from the Gray Level Co-occurrence Matrix (GLCM), e.g. contrast, entropy:
     $$
     \text{Contrast} = \sum_{i,j} (i-j)^2 \, \text{GLCM}(i,j)
     $$
   - Calculate backscatter ratios:
     $$
     R = \frac{\sigma^0_{HH}}{\sigma^0_{HV}}
     $$

9. **Classification of Ice Types:**
   - Use extracted features $[\sigma^0_{HH}, \sigma^0_{HV}, R, \text{Texture}_1, ...]$ as input vectors $\vec{x}$ to a classifier:
     $$
     \text{Class} = \mathrm{ML\,or\,CNN}(\vec{x})
     $$
   - Output categories: open water, new ice, first-year ice, multi-year ice.


## Ship detection workflow (GRD)

1. Preprocessing of GRD data:
   - Apply precise orbit file corrections to improve geolocation accuracy.
   - Radiometrically calibrate the data to convert pixel values to backscatter coefficients (sigma0).
   - Generally avoid strong speckle filtering at this stage to preserve small ship targets, as aggressive filtering can mask point-like objects.

2. Land-Sea Masking:
   - Mask out land areas using a detailed coastline shapefile or a Digital Elevation Model (DEM).
   - This focuses subsequent detection steps over the sea and prevents false positives from bright land targets.

3. Adaptive Thresholding (CFAR - Constant False Alarm Rate):
   - For each pixel, compute a local background mean or statistical distribution (in a moving window).
   - Set a threshold based on the background, flagging pixels significantly brighter than their surroundings as potential ship candidates.
   - Variants include CA-CFAR (cell-averaging), OS-CFAR (ordered statistics), etc.

4. Object Discrimination:
   - Group connected detected pixels into distinct objects ("blobs").
   - Remove objects that are smaller than the expected ship size (likely noise) or larger than a plausible ship (likely land, islands, or bright artifacts).
   - Additional metrics such as shape, orientation, and backscatter amplitude can be used to further refine detections.

5. Exporting Results:
   - Extract the geolocated positions and estimated sizes/shapes of all detected ships.
   - Output may be provided in vector format (e.g., GeoJSON, shapefiles, or CSV with coordinates, bounding boxes, and ancillary info).

6. (Optional) Matching with AIS (Automatic Identification System) Records:
   - Some Sentinel-1 missions (C and D) carry AIS receivers, which can record transmitted vessel locations.
   - Match detected objects with AIS records based on time and proximity to validate detections, annotate ship identities, or assess non-cooperative traffic.


## Level-2 OCN (Ocean) Products

Sentinel-1's Level-2 Ocean (OCN) products are derived data outputs generated and preprocessed by ESA, removing the need for users to perform their own processing beyond data extraction and interpretation. These products are distributed in NetCDF format and contain geolocated, value-added physical measurements ready for analysis.

The main Level-2 OCN products include:

- **OWI (Ocean Wind Fields):**
    - Provides estimates of ocean surface wind speed (in m/s) and wind direction (in degrees) at each measurement cell.
    - Useful for marine meteorology, weather forecasting, and hazard monitoring.
    - Includes quality flags, uncertainty metrics, geolocation, and time stamps for each wind vector cell.
    - Generated from both Interferometric Wide (IW) and Extra Wide (EW) modes, and covers most open ocean areas observed by Sentinel-1.

- **OSW (Ocean Swell Spectra):**
    - Contains estimates of ocean swell parameters, such as swell height, period, and direction, derived from SAR wave mode (WV) and stripmap mode (SM).
    - Data is typically provided as wave spectra (energy density as a function of frequency and direction), allowing reconstruction of dominant wave fields and their long-distance propagation.
    - Used for oceanography, coastal studies, and maritime safety assessments.

- **RVL (Radial Surface Velocity):**
    - Delivers measurements of the radial component (towards or away from the sensor) of surface water velocity.
    - Derived using Doppler centroid estimations from the SAR signal.
    - Valuable for tracking surface currents, ocean circulation patterns, and rapidly changing conditions such as tides or river plumes.

These Level-2 products are accompanied by detailed metadata, including timestamps, quality indicators, source orbit details, and processing history, ensuring traceability and scientific utility.


## Choosing the right workflow

| Goal | Product | Workflow |
|---|---|---|
| Land cover, crops, soil moisture | IW GRD | GRD backscatter; multitemporal GRD time series |
| Flood extent | IW GRD | GRD backscatter; multitemporal GRD time series |
| Earthquake or volcano displacement | IW SLC | Differential InSAR |
| Harvest, damage, urban change | IW SLC | Interferometric coherence |
| Slow subsidence (mm/yr) | IW SLC stack | InSAR time series (SBAS and PS-InSAR) |
| Vegetation structure features | IW SLC | Dual-pol polarimetric decomposition |
| Sea ice | EW GRDM | EW mode sea ice |
| Ships, oil spills | IW/EW GRD | Ship detection |
| Ocean wind and waves | Level-2 OCN | Level-2 OCN products |


## Sentinel-1 Processing with GeoLattice


Below are short descriptions of commonly-used functions for processing Sentinel-1 imagery:

- **search_sentinel1()**: Searches for Sentinel-1 data by specifying criteria such as date range, area of interest, polarization, and product type.
- **download_sentinel1()**: Downloads Sentinel-1 products from various repositories (e.g., ASF, Copernicus Open Access Hub) using product IDs or metadata.
- **preprocess_grd()**: Performs preprocessing of Sentinel-1 GRD data, such as radiometric calibration, geocoding, and terrain correction.
- **rtc_processing()**: Generates Radiometrically Terrain Corrected (RTC) products from GRD data, removing terrain-induced distortions.
- **insar_coregister()**: Coregisters a stack or pair of Sentinel-1 SLC images, aligning them for interferometric processing.
- **generate_interferogram()**: Generates interferograms from coregistered SLC pairs to map phase differences due to ground deformation.
- **time_series_insar()**: Processes a stack of SLC images to generate time series of ground displacement (for example, using MintPy or Dolphin).
- **visualize_sar()**: Loads and displays Sentinel-1 SAR products in analysis-ready formats, such as amplitude, coherence, or displacement.
- **apply_speckle_filter()**: Reduces speckle noise in SAR imagery using spatial filtering techniques.
- **export_geotiff()**: Saves Sentinel-1 processed products (RTC, interferogram, etc.) to GeoTIFF format for further geospatial analysis.

These functions may be accessible via packages like `geolattice-sentinel1`, `pyroSAR`, `sarsen`, or custom scripts, depending on the workflow.


## Summary and Conclusion

In this section, we introduced Sentinel-1 satellite data and its key imaging modes, highlighting their relevance to a wide range of remote sensing applications. We provided a guide to selecting the appropriate Sentinel-1 product and workflow based on specific research or monitoring goals, such as land cover mapping, disaster detection, and ground deformation analysis. 

We also outlined common processing steps and Python-based tools used in handling Sentinel-1 data, including data searching, downloading, preprocessing, interferometric analysis, visualization, and export. These tools support efficient and reproducible workflows, making Sentinel-1 an accessible resource for geospatial analysis. 

By understanding the fundamentals of Sentinel-1 imagery, its workflows, and useful processing functions, users can confidently begin to leverage radar data for a wide variety of scientific and practical applications.


## Additional Resources


- [ESA Sentinel Online: Sentinel-1 Mission](https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-1)
- [ASF DAAC Sentinel-1 Guide](https://asf.alaska.edu/information/data-sets/sar-data-sets/sentinel-1/)
- [Sentinel-1 User Guide](https://sentinel.esa.int/web/sentinel/user-guides/sentinel-1-sar)
- [Copernicus Open Access Hub (Data Search & Download)](https://scihub.copernicus.eu/)
- [ASF Vertex (Search & Download Portal)](https://search.asf.alaska.edu/#/)
- [pyroSAR Documentation](https://pyrosar.readthedocs.io/en/latest/)
- [ESA SNAP (Sentinel Application Platform)](https://step.esa.int/main/toolboxes/snap/)
- [MintPy Documentation (InSAR Time Series)](https://mintpy.readthedocs.io/en/latest/)
- [Dolphin InSAR Processing](https://github.com/opera-adt/dolphin)
- [SAR Handbook: Comprehensive Methodologies for Forest Monitoring and Biomass Estimation](https://servirglobal.net/Portals/0/Documents/Publications/SAR-HANDBOOK-COMPREHENSIVE-METHODOLOGIES-FOR-FOREST-MONITORING-AND-BIOMASS-ESTIMATION.pdf)
#


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.